# World Models 1: CarRacing, Reinforcement Learning, and Rollout Data

## Case study and theory

A world model is a learned compact simulator: it compresses observations, predicts action-conditioned transitions, and lets a policy plan in an imagined environment. We use CarRacing as the motivating continuous-control task. It is naturally a partially observable Markov decision process: the agent sees an image $o_t$, chooses steering/throttle/brake $a_t$, receives reward $r_t$, and seeks to maximize expected discounted return $E[\sum_t \gamma^t r_t]$.

The classic World Models recipe combines a visual variational autoencoder (VAE), a recurrent mixture-density network (MDN-RNN), and a controller optimized in the learned dynamics. This notebook covers the environment and the first prerequisite: collecting aligned observations, actions, rewards, terminal flags, and episode IDs.

Gymnasium `CarRacing-v3` uses Box2D and may need extra dependencies. This notebook attempts to create it; if unavailable, a procedural road-scene simulator preserves the image/action data shapes. The fallback demonstrates the workflow, not official CarRacing physics or benchmark performance. Random rollouts are exploratory data, not a driving policy.

## 1. Setup and procedural fallback

The fallback renderer creates 64×64 RGB road frames. Steering changes lateral position, throttle changes speed, and leaving the road incurs a penalty. The same array schema is used for both fallback and Gymnasium data.

**What this cell does:** imports dependencies, sets seeds, creates a shared data directory, and defines the renderer and fallback rollout generator.

In [ ]:
import random
from pathlib import Path
import numpy as np, torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
SEED=41; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT=Path("world-model-data"); ROOT.mkdir(exist_ok=True); DATA=ROOT/"car_racing_rollouts.npz"; SIZE=64

def render(pos,curve,speed):
 h=w=SIZE; a=np.zeros((h,w,3),np.uint8); y=np.linspace(0,1,h); a[:,:,1]=75; a[:,:,0]=30; a[:,:,2]=25
 center=w/2+(y-.2)*curve*23-pos*12; half=5+y*25
 for r in range(h):
  l=max(0,int(center[r]-half[r])); q=min(w,int(center[r]+half[r])); a[r,l:q]=[78,78,78]
  if l>0:a[r,max(0,l-2):l]=[225,225,210]
  if q<w:a[r,q:min(w,q+2)]=[225,225,210]
 cx=int(np.clip(w/2+pos*13,5,w-6)); a[h-13:h-7,cx-4:cx+4]=[220,50,45]; return a

def demo(n=100,horizon=40):
 obs=[]; acts=[]; rews=[]; dones=[]; eps=[]
 for ep in range(n):
  pos=random.uniform(-.4,.4); curve=random.uniform(-1,1); speed=random.random()*.6
  for t in range(horizon):
   act=np.array([random.uniform(-1,1),random.random(),random.random()*.12],np.float32); obs.append(render(pos,curve,speed)); acts.append(act)
   pos=np.clip(pos+.045*act[0],-1.5,1.5); speed=np.clip(speed+.025*act[1]-.06*act[2]-.01,0,1); off=abs(pos)>1
   rews.append(float(.15*speed-.4*off-.02)); dones.append(float(off or t==horizon-1)); eps.append(ep)
   if off:break
 return np.array(obs),np.array(acts),np.array(rews,np.float32),np.array(dones,np.float32),np.array(eps)


## 2. Environment and observation/action contract

Gymnasium CarRacing returns RGB images and a three-value continuous action: steering, gas, and brake. We resize real frames to 64×64. If Box2D or Gymnasium is unavailable, the procedural simulator is selected automatically. Set `USE_GYMNASIUM=False` to force the fallback.

**What this cell does:** tries to initialize the real environment and displays one observation.

In [ ]:
USE_GYMNASIUM=True
env=None
if USE_GYMNASIUM:
 try:
  import gymnasium as gym
  env=gym.make("CarRacing-v3",continuous=True,render_mode="rgb_array")
 except Exception as e: print("CarRacing unavailable; fallback:",type(e).__name__,str(e)[:120])
if env is not None: frame,_=env.reset(seed=SEED)
else: frame=render(.1,.4,.5)
plt.figure(figsize=(3,3)); plt.imshow(frame); plt.axis("off"); plt.title("Observation"); plt.show()
print("shape:",frame.shape,"action example: [steer, gas, brake]")


## 3. Collect random rollout data

A rollout records transition tuples $(o_t,a_t,r_t,d_t)$. Episode IDs let later notebooks build temporal windows without crossing reset boundaries. We cap steps to keep the demo quick. Random actions cover varied states but usually earn poor reward.

**What this cell does:** collects a bounded dataset from either environment and standardizes observations to 64×64.

In [ ]:
if env is None:
 observations,actions,rewards,dones,episode_ids=demo(120,40); source="procedural fallback"
else:
 from PIL import Image
 obs=[]; acts=[]; rews=[]; dns=[]; ids=[]
 for ep in range(8):
  ob,_=env.reset(seed=SEED+ep)
  for t in range(180):
   ac=env.action_space.sample(); nxt,r,term,trunc,_=env.step(ac); done=term or trunc
   obs.append(np.asarray(Image.fromarray(nxt).resize((SIZE,SIZE)))); acts.append(ac); rews.append(r); dns.append(float(done)); ids.append(ep)
   ob=nxt
   if done:break
 observations=np.stack(obs); actions=np.asarray(acts,np.float32); rewards=np.asarray(rews,np.float32); dones=np.asarray(dns,np.float32); episode_ids=np.asarray(ids); source="Gymnasium CarRacing-v3"
print(source,"| observations",observations.shape,"actions",actions.shape,"episodes",len(np.unique(episode_ids)))
print("reward range:",float(rewards.min()),float(rewards.max()))


## 4. Inspect and save the dataset

Inspect a reward trace and sample frames before training. The reward in the fallback is only a pedagogical proxy. We save a compressed NumPy archive so notebooks 02 and 03 can reuse the exact same aligned rollouts. Record environment versions and seeds for a reproducible experiment.

**What this cell does:** plots one episode, saves the rollout arrays, and closes the optional Gymnasium environment.

In [ ]:
ix=np.flatnonzero(episode_ids==np.unique(episode_ids)[0]); fig,ax=plt.subplots(1,2,figsize=(9,3))
ax[0].plot(rewards[ix]); ax[0].set(title="Episode reward",xlabel="step"); ax[0].grid()
for k,j in enumerate([ix[0],ix[len(ix)//2],ix[-1]]): ax[1].imshow(observations[j]); ax[1].axis("off")
plt.tight_layout(); plt.show()
np.savez_compressed(DATA,observations=observations,actions=actions,rewards=rewards,dones=dones,episode_ids=episode_ids)
print("saved",DATA.resolve(),"size MB",round(DATA.stat().st_size/1e6,2))
if env is not None:env.close()


## Key takeaways

- RL optimizes long-term reward through interaction; image-based driving is partially observable.
- A world model learns compact latent states and action-conditioned dynamics for planning in imagination.
- Keep observations, actions, rewards, terminal flags, and episode boundaries aligned.
- Random rollouts provide exploratory data but do not solve CarRacing.
- The fallback preserves the pipeline, not Box2D physics or benchmark validity.
- Next, train a VAE on the frames and an MDN-RNN on the resulting latent transitions.